In [1]:
import os
import warnings
os.chdir("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [2]:
df_listings = pd.read_csv('data/combined_csvs/listings_property_vals.csv')

In [3]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def create_city_level_clusters(df, target_cols, city_col='city', k_per_city=6):
    """
    Groups data by city, applies K-Means clustering (default k=6) locally to each city,
    assigns a globally unique cluster ID, and calculates target averages.
    """
    print(f"Clustering by city using k={k_per_city}...")
    
    # Store the processed dataframes for each city
    processed_cities = []
    
    # 1. Iterate through each city independently
    for city, city_df in df.groupby(city_col):
        n_samples = len(city_df)
        
        # Safeguard: If a city has fewer listings than k_per_city, lower k
        current_k = min(k_per_city, n_samples)
        
        # If there's only 1 listing in the city, skip clustering logic for it
        if current_k < 2:
            city_df = city_df.copy()
            city_df['cluster_id'] = f"{city}_0"
            processed_cities.append(city_df)
            continue
            
        # 2. Extract and scale features (Spatial + Targets)
        feature_cols = ['latitude', 'longitude'] + target_cols
        X = city_df[feature_cols].fillna(0).values 
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # 3. Fit K-Means
        kmeans = KMeans(n_clusters=current_k, random_state=42, n_init=10)
        local_labels = kmeans.fit_predict(X_scaled)
        
        # 4. Create globally unique cluster IDs (e.g., "Miami_2")
        city_df = city_df.copy()
        city_df['cluster_id'] = [f"{city}_{label}" for label in local_labels]
        
        processed_cities.append(city_df)
        
    # 5. Combine all city dataframes back into one
    result_df = pd.concat(processed_cities)
    
    # 6. Calculate the global averages for each unique cluster ID
    print(f"Aggregating averages for: {target_cols}")
    cluster_averages = result_df.groupby('cluster_id')[target_cols].mean()
    
    # Rename columns 
    rename_dict = {col: f"cluster_avg_{col}" for col in target_cols}
    cluster_averages = cluster_averages.rename(columns=rename_dict)
    
    # 7. Merge the averages back onto the main dataframe
    result_df = result_df.merge(cluster_averages, left_on='cluster_id', right_index=True, how='left')
    
    return result_df


In [4]:
# Define the columns you want to average
targets = ['ttm_revenue', 'gross_yield', 'ttm_occupancy']

# Run the function on your dataframe (assuming it's called df_listings)
df_clustered = create_city_level_clusters(df_listings, target_cols=targets, city_col='city', k_per_city=6)

Clustering by city using k=6...
Aggregating averages for: ['ttm_revenue', 'gross_yield', 'ttm_occupancy']


In [6]:
df_clustered.head()

,listing_id,listing_name,description,listing_type,room_type,cover_photo_url,photos_count,photo_urls,host_id,host_name,...,city,Geocodio Address Line 1,zipcode,property_val,SizeRank,gross_yield,cluster_id,cluster_avg_ttm_revenue,cluster_avg_gross_yield,cluster_avg_ttm_occupancy
1710,155305,Cottage! BonPaul + Sharky's Hostel,West Asheville Cottage within walking distance...,Entire guesthouse,entire_home,https://a0.muscache.com/im/pictures/hosting/Ho...,8.0,https://a0.muscache.com/im/pictures/hosting/Ho...,746673,BonPaul,...,asheville,37 Sand Hill Rd,28806,301336.204710,1486.0,0.066082,asheville_4,31292.054945,0.083661,0.545538
1711,197263,Tranquil Room & Private Bath,"This comfy, peaceful room has off-street parki...",Private room in home,private_room,https://a0.muscache.com/im/pictures/miso/Hosti...,5.0,https://a0.muscache.com/im/pictures/miso/Hosti...,961396,Timothy,...,asheville,167 Pisgah View Rd,28806,301336.204710,1486.0,0.008330,asheville_3,10147.750000,0.029261,0.200519
1712,209068,Terrace Cottage,Located in one of Asheville's oldest historic ...,Entire guest suite,entire_home,https://a0.muscache.com/im/pictures/1829924/9f...,18.0,https://a0.muscache.com/im/pictures/1829924/9f...,1029919,Kevin,...,asheville,2 Garden Ter,28804,255365.058601,4917.0,0.011149,asheville_2,8360.178082,0.023918,0.167753
1713,246315,Asheville Dreamer's Cabin,NaN,Private room in cabin,private_room,https://a0.muscache.com/im/pictures/5908224/37...,21.0,NaN,1292070,Annie,...,asheville,49 Trinity Chapel Rd,28805,264457.472429,6191.0,0.010047,asheville_2,8360.178082,0.023918,0.167753
1714,314540,Asheville Urban Farmhouse Entire Home 4.6 mi t...,NaN,Entire home,entire_home,https://a0.muscache.com/im/pictures/hosting/Ho...,56.0,NaN,381660,Tom,...,asheville,18 Pisgah View Rd,28806,558827.772716,1486.0,0.070159,asheville_3,10147.750000,0.029261,0.200519


In [8]:
# Save the dataframe to a CSV in your current working directory
df_clustered.to_csv('data/combined_csvs/listings_clustered.csv', index=False)